# Tester_branch에서 성능 확인 진행

In [ ]:
# 라이브러리 및 데이터 불러오기

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from sklearn.datasets import load_wine

from sklearn.model_selection import train_test_split, GridSearchCV

import matplotlib.pyplot as plt

wine = load_wine()

''' 데이터 코드를 작성해주세요 '''
df = pd.DataFrame(data = wine.data, columns = wine.feature_names)
df['target'] = wine.target

target = df['target']
feature = df.drop(columns='target')

X_train, X_test, y_train, y_test = train_test_split(feature, target, test_size=0.2, random_state=42)
# feature로 사용할 데이터에서는 'target' 컬럼을 drop합니다.
# target은 'target' 컬럼만을 대상으로 합니다.
# X, y 데이터를 test size는 0.2, random_state 값은 42로 하여 train 데이터와 test 데이터로 분할합니다.

# DT 모델링 성능

In [ ]:
''' 코드를 작성해주세요 '''
''' 코드를 작성해주세요 '''
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

param_grid = {
    "criterion": ['gini', 'entropy']
    , "max_depth": [2, 3, 4, 5]
    , "min_samples_split": [2, 5, 10]
    , "min_samples_leaf": [1, 2, 4]
}

# HPO 및 Fitting
clf_grid = DecisionTreeClassifier(random_state= 42)
# core
grid_search = GridSearchCV(clf_grid, param_grid, cv = 5, scoring='accuracy')
# HyperParameter를 찾고, 이걸 가지고 fitting 모두 수행
grid_search.fit(X_train, y_train)


# HPO 만들어진 모델의 정확도 계산
best_model_dt = grid_search.best_estimator_
y_pred_tuned_dt = best_model_dt.predict(X_test)

accuracy_score_dt = accuracy_score(y_test, y_pred_tuned_dt)
classification_report_dt = classification_report(y_test, y_pred_tuned_dt)


<img src="../참고이미지/Git/DT_Modeling.png" width="80%">

# XGB 모델링 성능

In [ ]:
''' 코드를 작성해주세요 '''
from sklearn.preprocessing import LabelEncoder, StandardScaler
from xgboost import XGBClassifier
# from sklearn.metrics import accuracy_score, classification_report

# 레이블링 인코딩
label_encoder = LabelEncoder() 
y_train_encoded = label_encoder.fit_transform(y_train) 
y_test_encoded = label_encoder.transform(y_test)

xgb_model = XGBClassifier(random_state=42)
xgb_model.fit(X_train, y_train_encoded)

# 예측 및 레이블 디코딩
rf_y_pred_encoded = xgb_model.predict(X_test)
y_pred_xgb = label_encoder.inverse_transform(rf_y_pred_encoded)

params = {
    "max_depth" : [3, 5, 7, 9, 15],
    "learning_rate" : [0.1, 0.01, 0.001],
    "n_estimators": [50, 100, 200, 300]
}
# 하이퍼파라미터 최적화 
grid_search = GridSearchCV(estimator=xgb_model, param_grid=params, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train_encoded)

# 최적의 하이퍼파라미터의 학습
best_model_xgb = grid_search.best_estimator_

#테스트 데이터에 대한 예측
y_pred_encoded_xgb = best_model_xgb.predict(X_test)
y_pred_xgb = label_encoder.inverse_transform(y_pred_encoded_xgb)

accuracy_score_xgb = accuracy_score(y_test, y_pred_xgb)
classification_report_xgb = classification_report(y_test, y_pred_xgb)

<img src="../참고이미지/Git/XGB_Modeling.png" width="80%">`

# 성능 비교 시각화

In [ ]:
''' 코드를 작성해주세요 '''
print("================== DT 성능 확인 ==================")
print(f"DT accuracy : {accuracy_score_dt}")
print(classification_report_dt)

print("================== XGB 성능 확인 ==================")
print(f"XGB accuracy : {accuracy_score_xgb}")
print(classification_report_xgb)


accuracy_scores_dict = {'DT':accuracy_score_dt, 'XGB':accuracy_score_xgb}
accuracy_score_df = pd.DataFrame(accuracy_scores_dict, index=[0])
accuracy_score_df.plot(kind='bar')


<img src="../참고이미지/Git/성능_비교_시각화.png" width="40%">